In [1]:
!pip install -q accelerate transformers torch langdetect


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\furyd\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Importing the necessary libraries
import re
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langdetect import detect

C:\Users\furyd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langdetect import detect

# Initialize Qwen Powered Chatbot and Translation Engine
class NotebookTranslator:
    def __init__(self):
        print("Loading Ultra-Fast Qwen 0.5B Engine...")
        # NO HUGGING FACE TOKEN OR API LOGIN NEEDED FOR THIS OPEN MODEL
        self.model_name = "Qwen/Qwen2.5-0.5B-Instruct"  
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        
        # Load the model directly to CPU (Fits under 1GB RAM naturally!)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype=torch.float32, 
            device_map="cpu"  
        )
        
        self.LANG_MAP = {
            "en": "English",
            "fr": "French",
            "es": "Spanish",
            "de": "German",
            "da": "Danish",
            "zh": "Chinese (Simplified)"
        }

    def detect_language(self, text):
        try:
            if not text or not any(char.isalpha() for char in text):
                return "en"
            return detect(text)
        except Exception:
            return "en" 

    def translate(self, text, source='auto', target='en'):
        target_lang_name = self.LANG_MAP.get(target, "English")
        
        prompt_transl = (
            f"Translate the following text strictly into {target_lang_name}. "
            f"Provide only the direct translation output with no introductory text, notes, or explanations.\n\n"
            f"Text to translate:\n\"{text}\""
        )
        
        messages = [{"role": "user", "content": prompt_transl}]
        formatted_prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        inputs = self.tokenizer(formatted_prompt, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            generated_tokens = self.model.generate(
                **inputs,
                max_new_tokens=250,
                temperature=0.1,  
                do_sample=False
            )
            
        input_length = inputs.input_ids.shape[1]
        raw_output_text = self.tokenizer.decode(generated_tokens[0][input_length:], skip_special_tokens=True)
        return raw_output_text.strip().strip('"')

# Instantiate the translator module using Qwen
translator = NotebookTranslator()

# Link your generator pipeline seamlessly using the same CPU memory block
print("Initializing shared generator pipeline...")
generator = pipeline(
    "text-generation", 
    model=translator.model, 
    tokenizer=translator.tokenizer
)
print("Middle-ground Multilingual Engine ready and optimized for speed!")

Loading Ultra-Fast Qwen 0.5B Engine...


Device set to use cpu


Initializing shared generator pipeline...
Middle-ground Multilingual Engine ready and optimized for speed!


In [4]:
def extract_questions(text):
    return re.findall(r'\d+\..*?\?', text) or [text]

def generate_ai_response(prompt_text, max_new_tokens=250):
    # STRUCTURE
    messages = [
        {"role": "system", "content": "You are a helpful, direct assistant. Answer the user's questions accurately and concisely without any fluff."},
        {"role": "user", "content": prompt_text}
    ]
    
    # 2. Inject the native structural chat tags into the text string
    formatted_prompt = translator.tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # 3. Tokenize and move to the CPU device
    inputs = translator.tokenizer(formatted_prompt, return_tensors="pt").to(translator.model.device)
    
    # 4. Generate the response with do_sample=True to completely fix the flag warning
    with torch.no_grad():
        generated_tokens = translator.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,   
            top_p=0.9,         
            do_sample=True     # Allows the chatbot to think creatively and talk normally
        )
        
    # 5. Decode only the new parts written by the AI
    input_length = inputs.input_ids.shape[1]
    response_text = translator.tokenizer.decode(
        generated_tokens[0][input_length:], 
        skip_special_tokens=True
    )
    
    return response_text.strip()

In [5]:
print("\n" + "="*40)
enquiry_text = input("Enter your enquiry: ").strip()  
# Added target language preference input
target_pref = input("Enter desired target language code (e.g., 'en', 'fr', 'es', 'de'): ").strip().lower()
print("="*40)

In [6]:
# DETECT SOURCE LANGUAGE 
detect_lang = translator.detect_language(enquiry_text)
translated_enq = translator.translate(enquiry_text, source=detect_lang, target=target_pref)

# Print results
print(f"Original Enquiry: {enquiry_text}")
print(f"Detected Language: {detect_lang.upper()}")
print(f"Translated Response Target ({target_pref.upper()}): {translated_enq}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Original Enquiry: Hi I'm Daniel! Would you mind telling me what the capital of Angola or Zimbabwe is?
Detected Language: EN
Translated Response Target (DE): Hallo, ich bin Daniel! Möchten Sie bitte sagen, was die Hauptstadt Angola oder Zimbabwe ist?


In [7]:
# QUESTION SEARCH
def contains_question(text):
    return '?' in text.strip()

print(f"Contains Question? {contains_question(enquiry_text)}")

Contains Question? True


In [8]:
# Generate and display responses

# Enquiry in English
enquiry_en = translator.translate(enquiry_text, source=detect_lang, target='en')

# Response in English
ai_response = generate_ai_response(enquiry_en, max_new_tokens=150)
print(f"AI Response (Internal English): {ai_response}\n")

# Re-translates back to user's preferred language
response_transl = translator.translate(ai_response, source='en', target=target_pref)
print(f"AI Chatbot Response ({target_pref.upper()}): {response_transl}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


AI Response (Internal English): Angola has the capital city of Luanda, while Zimbabwe has Harare as its capital city.

AI Chatbot Response (DE): Angola hat das Hauptstadt Luanda als Hauptstadt der Republik, während Zimbabwe die Stadt Harare als Hauptstadt hat.
